In [ ]:
# transformers がすでにインストールされているかを確認し、
# 未インストールの場合のみインストールする
import importlib.util
import sys
import subprocess

# importlib.util.find_spec("transformers") は、
# Python環境内に transformers パッケージが存在するかを確認する
# None の場合は未インストール、None でなければインストール済み
if importlib.util.find_spec("transformers") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "transformers[ja,sentencepiece,torch]"]
    )
else:
    print("transformers はすでにインストールされています。")


In [3]:
import os
import warnings

# Hugging Face Hub からの認証なしリクエストに関する警告など、
# 実行結果に不要な警告を表示しないようにする。
warnings.filterwarnings("ignore")

# Hugging Face Hub の進捗バーを無効化する。
# モデルの重みダウンロード時に表示される tqdm の progress bar を抑制する。
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Transformers 側のログ出力を error のみに制限する。
# これにより、モデル読み込み時の warning / info レベルの出力を抑える。
from transformers.utils import logging

logging.set_verbosity_error()

from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 事前学習済み・ファインチューニング済みモデルによるテキスト分類を行う。
#
# pipeline は、トークナイズ・モデルへの入力・推論・出力整形をまとめて行う高水準API。
# ここでは感情極性分類、つまり文章がポジティブかネガティブかを判定するモデルを使う。
text_classification_pipeline = pipeline(
    # text-classification を明示することで、
    # この pipeline が文章分類タスク用であることをコード上から分かりやすくする。
    "text-classification",
    # llm-book/bert-base-japanese-v3-marc_ja は、
    # 日本語BERTを日本語レビュー分類データセットでファインチューニングしたモデル。
    #
    # BERTはTransformer Encoderをベースにしたモデルであり、
    # 文中の各トークンがSelf-Attentionによって他のトークンとの関係を考慮しながら表現される。
    #
    # テキスト分類では、文章全体の意味を表すベクトルを分類層に入力し、
    # positive / negative などのラベルごとのスコアを計算する。
    model="llm-book/bert-base-japanese-v3-marc_ja",
)

# ポジティブ寄りの文章を用意する。
# 「感動する」という語が含まれており、文全体として肯定的な評価を表している。
positive_text = "世界には言葉がわからなくても感動する音楽がある。"

# ネガティブ寄りの文章を用意する。
# 「ひどい」という語が含まれており、文全体として否定的な評価を表している。
negative_text = "世界には言葉がでないほどひどい音楽がある。"

# 複数の文章をリストにまとめて pipeline に渡す。
#
# 1文ずつ推論することもできるが、複数文をまとめて渡すことで、
# 入力と出力の対応関係を管理しやすくなる。
texts = [
    positive_text,
    negative_text,
]

# texts の各文章について感情極性を予測する。
#
# 内部では以下の処理が行われる。
#
# 1. 文章をトークンに分割する
# 2. トークンをID列に変換する
# 3. BERTに入力してSelf-Attentionにより文脈表現を得る
# 4. 分類層で各ラベルのスコアを計算する
# 5. 最も確率の高いラベルとそのスコアを返す
results = text_classification_pipeline(texts)

# 余計な説明文を出さず、分類結果だけを表示する。
for result in results:
    print(result)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20540.78it/s]


{'label': 'positive', 'score': 0.9993619322776794}
{'label': 'negative', 'score': 0.9636250138282776}


In [4]:
from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 自然言語推論 Natural Language Inference, NLI を行う。
#
# NLI は、2つの文の論理的な関係を分類するタスク。
# 主に以下の3種類のラベルに分類される。
#
# entailment:
#   1つ目の文が正しいとき、2つ目の文も必ず正しいと考えられる関係。
#   日本語では「含意」と呼ばれる。
#
# contradiction:
#   1つ目の文が正しいとき、2つ目の文は成り立たない関係。
#   日本語では「矛盾」と呼ばれる。
#
# neutral:
#   1つ目の文だけでは、2つ目の文が正しいとも間違いとも判断できない関係。
#   日本語では「中立」と呼ばれる。
#
# 例:
#   前提文: 「人が犬を散歩させている」
#   仮説文: 「人が動物と一緒にいる」
#   この場合、犬は動物なので entailment になりやすい。
#
#   前提文: 「人が犬を散歩させている」
#   仮説文: 「人が一人で歩いている」
#   この場合、犬と一緒にいるため contradiction になりやすい。
#
#   前提文: 「人が犬を散歩させている」
#   仮説文: 「その人は公園にいる」
#   この場合、公園かどうかは前提文だけでは分からないため neutral になりやすい。
nli_pipeline = pipeline(
    # llm-book/bert-base-japanese-v3-jnli は、
    # 日本語BERTを日本語自然言語推論データセット JNLI で
    # ファインチューニングしたモデル。
    #
    # BERT は Transformer Encoder に基づくモデルで、
    # Self-Attention によって文中の各トークンが他のトークンを参照しながら
    # 文脈を考慮した表現を獲得する。
    #
    # NLIでは、2つの文をまとめてBERTに入力する。
    # 一般にBERTでは、文ペアを以下のような形で扱う。
    #
    #   [CLS] 文1 [SEP] 文2 [SEP]
    #
    # [CLS] トークンに対応する出力ベクトルを文ペア全体の表現として使い、
    # その表現を分類層に入力して entailment / contradiction / neutral を予測する。
    model="llm-book/bert-base-japanese-v3-jnli"
)

# 前提文 premise を用意する。
#
# NLIでは、1つ目の文を「前提文」と呼ぶ。
# この文が事実として成り立っていると仮定したとき、
# 2つ目の文が論理的にどういう関係にあるかを判定する。
text = "二人の男性がジェット機を見ています"

# 含意関係になりやすい仮説文 hypothesis を用意する。
#
# 前提文:
#   「二人の男性がジェット機を見ています」
#
# 仮説文:
#   「ジェット機を見ている人が二人います」
#
# 前提文が正しければ、
# 「ジェット機を見ている人が二人いる」ことも自然に成り立つ。
# そのため、この文ペアは entailment と判定されることが期待される。
entailment_text = "ジェット機を見ている人が二人います"

# text と entailment_text の論理関係を予測する。
#
# pipeline に文ペアを入力するときは、
# {"text": 前提文, "text_pair": 仮説文} の辞書形式で渡す。
#
# 内部では、BERTのTokenizerが2文をまとめてトークン化し、
# token_type_ids によって「どこまでが1文目で、どこからが2文目か」を区別する。
#
# その後、BERTが文ペア全体の意味関係をSelf-Attentionで捉え、
# 最終的に3クラス分類としてラベルとスコアを返す。
print(nli_pipeline({"text": text, "text_pair": entailment_text}))

# 矛盾関係になりやすい仮説文を用意する。
#
# 前提文:
#   「二人の男性がジェット機を見ています」
#
# 仮説文:
#   「二人の男性が飛んでいます」
#
# 前提文では、男性たちは「ジェット機を見ている」と述べられている。
# 一方で、仮説文では男性たち自身が「飛んでいる」と述べられている。
#
# 通常の解釈では、
# 「ジェット機を見ている男性」と「男性自身が飛んでいる」は同時に成立しにくい。
# そのため、この文ペアは contradiction と判定されることが期待される。
contradiction_text = "二人の男性が飛んでいます"

# text と contradiction_text の論理関係を予測する。
#
# NLIモデルは、単語の一致だけではなく、
# 「見る」と「飛ぶ」の意味的な違いや、
# 主体が誰であるかを文脈的に考慮して分類する。
#
# ただし、モデルはあくまで学習データから得た統計的パターンに基づいて予測するため、
# 人間の論理推論と完全に一致するとは限らない。
print(nli_pipeline({"text": text, "text_pair": contradiction_text}))

# 中立関係になりやすい仮説文を用意する。
#
# 前提文:
#   「二人の男性がジェット機を見ています」
#
# 仮説文:
#   「2人の男性が、白い飛行機を眺めています」
#
# 「2人の男性が飛行機を見ている」という点は前提文と近い。
# しかし、前提文ではジェット機の色が「白い」とは述べられていない。
#
# そのため、前提文だけから
# 「白い飛行機である」とまでは断定できない。
# このように、正しいとも間違いとも言い切れない場合は neutral になりやすい。
neutral_text = "2人の男性が、白い飛行機を眺めています"

# text と neutral_text の論理関係を予測する。
#
# この例では、かなり意味が近い文同士だが、
# 「白い」という追加情報が前提文にないため、
# 厳密な自然言語推論では entailment ではなく neutral とみなされる可能性がある。
#
# NLIでは、このような「前提文に書かれていない追加情報」を
# どのように扱うかが重要になる。
print(nli_pipeline({"text": text, "text_pair": neutral_text}))


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24261.97it/s]


{'label': 'entailment', 'score': 0.9958741068840027}
{'label': 'contradiction', 'score': 0.9922866821289062}
{'label': 'neutral', 'score': 0.9821659922599792}
